# 3D Alignment Analysis Pipeline

## Overview
This notebook provides a complete and robust pipeline for analyzing the quality of synthetic 3D data generated by Blender. It automates the process of aligning generated point clouds to a reference model and calculating the error metrics.

### Key Features:
1.  **Automated Dependency Management:** Checks and installs required libraries automatically.
2.  **Structured Excel Reporting:** Generates detailed reports per batch, including Overview sheets and per-sensor statistics.
3.  **Robust Alignment Engine:** Utilizes a multi-stage alignment process (RANSAC Global Registration followed by ICP Local Refinement).
4.  **Correct Ground Truth Comparison:** Calculates error based on relative movement ($T_{relative} = T_{current} \cdot T_{ref}^{-1}$) to handle absolute coordinate differences correctly.
5.  **Advanced Visualization:** Includes tools to generate MP4/GIF animations of the alignment process and static 3D overlays.
6.  **Flexible Execution:** Can process the entire dataset or target specific subfolders.

## 1. Environment Setup & Dependency Management

This section handles the importation of necessary Python libraries. It includes a custom function `install_and_import` that checks if a package is installed. If a package is missing, it attempts to install it via `pip` before importing it into the global namespace.

**Libraries used:**
* `open3d`: For 3D point cloud processing and visualization.
* `pandas`: For data manipulation and reading/writing CSV/Excel files.
* `numpy`: For numerical operations and matrix algebra.
* `openpyxl`: The engine required by Pandas to write Excel files.
* `imageio`: For generating GIF and MP4 animations.
* `opencv-python` (cv2): For drawing text overlays on images.
* `ipython`: For display utilities within the notebook.

In [ ]:
import subprocess
import sys
import importlib
import os
import time
import math
import copy
import ast

# Configuration List: (Pip Package Name, Import Module Name, Alias)
required_libraries = [
    ("openpyxl", "openpyxl", None),       # Excel support
    ("imageio", "imageio", None),         # GIF/Video support
    ("imageio[ffmpeg]", "imageio", None), # FFmpeg plugin for MP4
    ("opencv-python", "cv2", None),       # Text overlays
    ("open3d", "open3d", "o3d"),          # 3D Point Clouds
    ("pandas", "pandas", "pd"),           # Data analysis
    ("numpy", "numpy", "np"),             # Math
    ("ipython", "IPython", None)          # Notebook display
]

def install_and_import(package_name, import_name, alias=None):
    """
    Checks if a library is installed. If not, installs it using pip.
    Then imports it into the global namespace with the specified alias.
    """
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Library '{import_name}' not found. Installing '{package_name}'...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
            print(f"Successfully installed: {package_name}")
        except subprocess.CalledProcessError:
            print(f"ERROR: Failed to install {package_name}. Please install manually.")
            return

    try:
        module = importlib.import_module(import_name)
        if alias:
            globals()[alias] = module
        else:
            globals()[import_name] = module
    except ImportError as e:
        print(f"Critical Error importing {import_name}: {e}")

print("--- Checking and Loading Libraries ---")
for pkg, mod, alias in required_libraries:
    install_and_import(pkg, mod, alias)

from IPython.display import display, Markdown
print("All libraries loaded and ready.")

## 2. Global Configuration

Here we define the global settings for the pipeline.

* **`PROJECT_DIR`**: The current working directory of this notebook.
* **`DATA_ROOT`**: The path where the generated batches are stored. Adjust this if your folder structure changes.
* **`REPORT_FILE`**: The name of the resulting Excel file.
* **`VOXEL_SIZE`**: A critical parameter for the alignment algorithm. It determines the resolution for the downsampling process used in Global Registration (RANSAC). A value of `0.05` means points are merged into a 5cm grid.

In [ ]:
# --- PATH CONFIGURATION ---
PROJECT_DIR = os.getcwd()
DATA_ROOT = os.path.join(PROJECT_DIR, "..", "Blender_Generated_Data")
REPORT_FILE = os.path.join(DATA_ROOT, "Alignment_Report_Final.xlsx")

# --- ALGORITHM CONFIGURATION ---
VOXEL_SIZE = 0.05

print(f"Data Root set to: {DATA_ROOT}")

## 3. Data Loading & Math Helper Functions

This block contains low-level utility functions used throughout the pipeline.

### Functions:
* **`load_csv_pcd(filepath)`**: Reads a CSV file containing X, Y, Z coordinates and converts it into an Open3D PointCloud object. It includes error handling for missing or empty files.
* **`parse_ground_truth_matrix(row)`**: Extracts the flattened 4x4 matrix data (columns `m00` through `m33`) from a Pandas DataFrame row and reshapes it into a numpy matrix.
* **`matrix_to_str(matrix)`** & **`str_to_matrix(mat_str)`**: Helpers to serialize numpy matrices to strings for Excel storage and retrieve them back.
* **`get_angular_error(R_est, R_gt)`**: Calculates the geodesic distance (angle in degrees) between an estimated rotation matrix and the ground truth rotation matrix.
* **`get_translation_error(t_est, t_gt)`**: Calculates the Euclidean distance (in meters) between two translation vectors.
* **`preprocess_point_cloud(pcd, voxel_size)`**: Prepares a point cloud for RANSAC. It performs downsampling, estimates normals (required for features), and computes Fast Point Feature Histograms (FPFH).

In [ ]:
def load_csv_pcd(filepath):
    """Loads XYZ data from a CSV file into an Open3D PointCloud."""
    try:
        if not os.path.exists(filepath): return None
        df = pd.read_csv(filepath)
        if len(df) < 10: return None
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(df[['X', 'Y', 'Z']].values)
        return pcd
    except: return None

def parse_ground_truth_matrix(row):
    """Extracts 4x4 matrix from dataframe row."""
    try:
        cols = [f"m{r}{c}" for r in range(4) for c in range(4)]
        flat = row[cols].values.astype(float)
        return flat.reshape(4, 4)
    except:
        return np.eye(4)

def matrix_to_str(matrix):
    """Flattens a 4x4 matrix to a string for Excel storage."""
    return str(matrix.tolist())

def str_to_matrix(mat_str):
    """Parses a string back to a numpy matrix."""
    try:
        return np.array(ast.literal_eval(mat_str))
    except:
        return np.eye(4)

def get_angular_error(R_est, R_gt):
    R_diff = np.dot(R_est, R_gt.T)
    tr = np.clip(np.trace(R_diff), -1, 3)
    return math.degrees(math.acos((tr - 1) / 2))

def get_translation_error(t_est, t_gt):
    return np.linalg.norm(t_est - t_gt)

def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    if len(pcd_down.points) < 5: return None, None
    pcd_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
    try:
        pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
            pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))
    except RuntimeError: return None, None
    return pcd_down, pcd_fpfh

## 4. The Alignment Engine

This function contains the core logic for registering (aligning) two point clouds.

### `align_point_cloud`

**Process:**
1.  **Preprocessing:** The source cloud is downsampled and FPFH features are calculated.
2.  **Global Registration (RANSAC):** Uses feature matching to find a rough alignment. This step is robust against large rotations and local minima. It operates on the downsampled data.
3.  **Local Refinement (ICP):** Uses the Point-to-Plane ICP algorithm to refine the alignment. It starts from the RANSAC result and minimizes the distance between points and the target surface.

**Returns:**
* The final 4x4 transformation matrix.
* A statistics dictionary containing fitness scores, RMSE, and timing info.

In [ ]:
def align_point_cloud(source_pcd, target_pcd, target_down, target_fpfh, voxel_size):
    start_time = time.time()
    stats = {}
    
    # 1. Preprocess Source
    source_down, source_fpfh = preprocess_point_cloud(source_pcd, voxel_size)
    if source_down is None: return None, None
    
    # 2. RANSAC (Global)
    distance_threshold = voxel_size * 1.5
    ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh, True,
        distance_threshold,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False), 3, 
        [o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
         o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)],
        o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999)
    )
    ransac_time = time.time()
    
    # 3. ICP (Local)
    icp_threshold = voxel_size * 0.4
    source_pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30))
    target_pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size*2, max_nn=30))
    
    icp = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd, icp_threshold, ransac.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )
    end_time = time.time()
    
    # Stats
    stats['time_ransac'] = ransac_time - start_time
    stats['time_icp'] = end_time - ransac_time
    stats['time_total'] = end_time - start_time
    stats['fitness'] = icp.fitness
    stats['rmse'] = icp.inlier_rmse
    
    return icp.transformation, stats

## 5. Main Analysis Pipeline & Reporting

This section contains the logic to iterate through the data folders and generate the Excel report.

### `save_batch_excel`
Helper function to save a dataframe to a structured Excel file. It creates an 'Overview' sheet with averages and separate sheets per sensor containing detailed data tables.

### `run_analysis_pipeline`
**Description:** The master function that executes the analysis.
**Parameters:**
* `target_subfolder` (optional): If provided (e.g., "Test_1"), the analysis will run ONLY on that specific batch folder. If `None`, it runs on all folders in `DATA_ROOT`.

**Logic:**
1.  scans the directory structure.
2.  Locates the Ground Truth CSV.
3.  Sets the first file in the Ground Truth as the **Reference**.
4.  Loops through all other files, aligns them to the Reference, and calculates the error relative to the Ground Truth matrix.
5.  Saves the results.

In [ ]:
def save_batch_excel(batch_id, df_results):
    """Saves report. Structure: Overview | Sensor1 | Sensor2 ..."""
    filename = os.path.join(DATA_ROOT, f"Analysis_{batch_id}.xlsx")
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # 1. Overview Sheet
        overview = df_results.groupby(['Sensor', 'Position'])[['Fitness', 'Error_Rot_Deg', 'Error_Trans_M', 'Time_Total']].mean()
        overview.to_excel(writer, sheet_name='Overview')
        
        # 2. Hidden Raw Data
        df_results.to_excel(writer, sheet_name='Raw_Data', index=False)
        
        # 3. Sheets per Sensor
        sensors = df_results['Sensor'].unique()
        for sensor in sensors:
            sheet_name = str(sensor)[:30]
            sensor_data = df_results[df_results['Sensor'] == sensor]
            positions = sensor_data['Position'].unique()
            
            row_cursor = 0
            for pos in positions:
                pos_data = sensor_data[sensor_data['Position'] == pos]
                
                # Header for Position Table
                pd.DataFrame([f"POSITION: {pos}"]).to_excel(writer, sheet_name=sheet_name, startrow=row_cursor, index=False, header=False)
                row_cursor += 1
                
                # Table
                cols = ['Sample_ID', 'Fitness', 'RMSE', 'Error_Rot_Deg', 'Error_Trans_M', 'Time_Total', 'Matrix_GT', 'Matrix_Est']
                pos_data[cols].to_excel(writer, sheet_name=sheet_name, startrow=row_cursor, index=False)
                
                row_cursor += len(pos_data) + 3
                
    print(f"   -> Report saved: {filename}")

def run_analysis_pipeline(target_subfolder=None):
    print("Scanning directory for datasets...")
    if not os.path.exists(DATA_ROOT): return

    # Get list of batch folders
    all_batches = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)) and d != "Output"]
    
    # Filter if target_subfolder is specified
    if target_subfolder:
        if target_subfolder in all_batches:
            batches = [target_subfolder]
            print(f"Targeting specific batch: {target_subfolder}")
        else:
            print(f"Error: Target folder '{target_subfolder}' not found.")
            return
    else:
        batches = all_batches
    
    status_handle = display(Markdown("**Starting Analysis...**"), display_id=True)
    
    for batch_id in batches:
        batch_results = []
        batch_path = os.path.join(DATA_ROOT, batch_id)
        
        # Walk Sensors & Positions
        sensors = [d for d in os.listdir(batch_path) if os.path.isdir(os.path.join(batch_path, d))]
        for sens_id in sensors:
            s_path = os.path.join(batch_path, sens_id)
            positions = [d for d in os.listdir(s_path) if os.path.isdir(os.path.join(s_path, d))]
            
            for pos_id in positions:
                final_path = os.path.join(s_path, pos_id)
                gt_file = os.path.join(final_path, "ground_truth.csv")
                if not os.path.exists(gt_file): continue
                
                df_gt = pd.read_csv(gt_file, comment='#')
                if df_gt.empty: continue
                
                # Set Reference (Scan 0 of this folder)
                ref_file = df_gt.iloc[0]['filename']
                target_pcd = load_csv_pcd(os.path.join(final_path, ref_file))
                if target_pcd is None: continue
                target_down, target_fpfh = preprocess_point_cloud(target_pcd, VOXEL_SIZE)
                if target_down is None: continue

                status_handle.update(Markdown(f"**Processing:** {batch_id} | {sens_id} | {pos_id}"))
                
                for _, row in df_gt.iterrows():
                    if row['filename'] == ref_file: continue # Skip ref
                    
                    pcd = load_csv_pcd(os.path.join(final_path, row['filename']))
                    if pcd is None: continue
                    
                    est_matrix, stats = align_point_cloud(pcd, target_pcd, target_down, target_fpfh, VOXEL_SIZE)
                    if est_matrix is None: continue
                    
                    gt_matrix = parse_ground_truth_matrix(row)
                    try:
                        T_inv = np.linalg.inv(est_matrix)
                        rot_err = get_angular_error(T_inv[:3,:3], gt_matrix[:3,:3])
                        trans_err = get_translation_error(T_inv[:3,3], gt_matrix[:3,3])
                    except: rot_err, trans_err = 999, 999
                    
                    batch_results.append({
                        "Batch": batch_id, "Sensor": sens_id, "Position": pos_id,
                        "Sample_ID": row['sample_id'],
                        "Fitness": stats['fitness'], "RMSE": stats['rmse'],
                        "Error_Rot_Deg": rot_err, "Error_Trans_M": trans_err,
                        "Time_Total": stats['time_total'],
                        "Matrix_GT": matrix_to_str(gt_matrix),
                        "Matrix_Est": matrix_to_str(est_matrix)
                    })
        
        if batch_results:
            save_batch_excel(batch_id, pd.DataFrame(batch_results))
            
    status_handle.update(Markdown("### ✅ Analysis Complete!"))

In [ ]:
# Example 1: Run analysis on ALL folders
run_analysis_pipeline()

# Example 2: Run analysis ONLY on a specific subfolder (e.g. "Test_1")
# run_analysis_pipeline(target_subfolder="Test_1")

## 6. Visualization & Animation

These functions allow you to inspect the results visually or generate animations for presentations.

### `resolve_scan_paths`
Internal helper to convert a relative path string into absolute file paths and load the necessary Ground Truth data.

### `view_alignment_overlay`
**Description:** Opens a 3D window showing the Reference (Grey), the Source (Red), and the Calculated Alignment (Green). 
**Smart Feature:** It attempts to load the calculated transformation matrix from the generated Excel report. This means you don't have to re-run the calculation to see the result.

### `create_presentation_gif`
**Description:** Generates a high-quality video (MP4) and GIF showing the alignment process.
**Visuals:** Splits the screen into 4 views (Front, Right, Top, Isometric).
**Sequence:**
1.  Show High-Res input.
2.  Show Downsampling process.
3.  Show RANSAC global search (simulating attempts).
4.  Show Coarse ICP refinement.
5.  Show Fine ICP refinement (High Res).
6.  Display Final Error statistics.

In [ ]:
# --- TEXT OVERLAY HELPER ---
def draw_text_overlay(image, text_lines, bg_color=(0, 0, 0), text_color=(255, 255, 255)):
    if isinstance(text_lines, str): text_lines = [text_lines]
    h, w, _ = image.shape
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.0 
    thickness = 2
    line_height = 40
    padding = 20
    box_height = padding * 2 + len(text_lines) * line_height
    cv2.rectangle(image, (0, 0), (w, box_height), bg_color, -1)
    y = padding + 30
    for line in text_lines:
        (text_w, text_h), _ = cv2.getTextSize(line, font, font_scale, thickness)
        text_x = (w - text_w) // 2
        cv2.putText(image, line, (text_x, y), font, font_scale, text_color, thickness, cv2.LINE_AA)
        y += line_height
    return image

def resolve_scan_paths(scan_folder_rel, source_id, ref_id):
    full_path = os.path.join(DATA_ROOT, scan_folder_rel)
    gt_path = os.path.join(full_path, "ground_truth.csv")
    if not os.path.exists(gt_path):
        print(f"Error: Path not found {scan_folder_rel}"); return None, None, None, None
    df_gt = pd.read_csv(gt_path, comment='#')
    src_row = df_gt[df_gt['sample_id'] == source_id]
    ref_row = df_gt[df_gt['sample_id'] == ref_id]
    if src_row.empty or ref_row.empty: print("Error: ID not found."); return None, None, None, None
    src_pcd = load_csv_pcd(os.path.join(full_path, src_row.iloc[0]['filename']))
    ref_pcd = load_csv_pcd(os.path.join(full_path, ref_row.iloc[0]['filename']))
    parts = scan_folder_rel.replace("\\", "/").split("/")
    batch = parts[0] if len(parts) > 0 else ""
    pos = parts[2] if len(parts) > 2 else ""
    return src_pcd, ref_pcd, batch, pos

def view_alignment_overlay(scan_folder, source_id, ref_id=0):
    """Shows static overlay using matrix from Excel."""
    src_pcd, ref_pcd, batch, pos = resolve_scan_paths(scan_folder, source_id, ref_id)
    if src_pcd is None: return
    excel_path = os.path.join(DATA_ROOT, f"Analysis_{batch}.xlsx")
    transform = np.eye(4)
    if os.path.exists(excel_path):
        try:
            df_raw = pd.read_excel(excel_path, sheet_name='Raw_Data')
            match = df_raw[(df_raw['Position'] == pos) & (df_raw['Sample_ID'] == source_id)]
            if not match.empty:
                transform = str_to_matrix(match.iloc[0]['Matrix_Est'])
        except: pass
    ref_pcd.paint_uniform_color([0.6, 0.6, 0.6]) # Grey
    src_pcd.transform(transform)
    src_pcd.paint_uniform_color([1, 0, 0])       # Red
    o3d.visualization.draw_geometries([ref_pcd, src_pcd], window_name=f"Overlay ID {source_id}", width=1000, height=800)

def create_presentation_gif(scan_folder, source_id, ref_id=0, save_gif=True, save_mp4=True, fps=10, pause_time=1.0):
    """Generates a storytelling GIF (High Res, Wide View, Fixed Math)."""
    src_pcd_high, ref_pcd_high, batch, pos = resolve_scan_paths(scan_folder, source_id, ref_id)
    if src_pcd_high is None: return
    
    # Load GT for stats
    full_path = os.path.join(DATA_ROOT, scan_folder)
    gt_path = os.path.join(full_path, "ground_truth.csv")
    df_gt = pd.read_csv(gt_path, comment='#')
    gt_row = df_gt[df_gt['sample_id'] == source_id].iloc[0]
    gt_matrix = parse_ground_truth_matrix(gt_row)

    print(f"Generating Animation for {scan_folder}...")
    ref_color = [0.6, 0.6, 0.6]; src_color = [1.0, 0.0, 0.0]
    ref_pcd_high.paint_uniform_color(ref_color); src_pcd_high.paint_uniform_color(src_color)
    
    # Pre-calc RANSAC
    src_down, s_fpfh = preprocess_point_cloud(src_pcd_high, VOXEL_SIZE)
    ref_down, t_fpfh = preprocess_point_cloud(ref_pcd_high, VOXEL_SIZE)
    src_down.paint_uniform_color(src_color); ref_down.paint_uniform_color(ref_color)
    
    ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        src_down, ref_down, s_fpfh, t_fpfh, True, VOXEL_SIZE * 1.5,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False), 3, 
        [o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
         o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(VOXEL_SIZE * 1.5)],
        o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999))
    ransac_final_T = ransac.transformation
    
    vis = o3d.visualization.Visualizer()
    vis.create_window(width=1000, height=1000, visible=True)
    vis.add_geometry(ref_pcd_high); vis.add_geometry(src_pcd_high)
    ctr = vis.get_view_control()
    
    def capture_frame(text_lines):
        views = [
            {'lookat': [0,0,0], 'front': [0, -1, 0], 'up': [0, 0, 1]}, 
            {'lookat': [0,0,0], 'front': [1, 0, 0],  'up': [0, 0, 1]}, 
            {'lookat': [0,0,0], 'front': [0, 0, 1],  'up': [0, 1, 0]}, 
            {'lookat': [0,0,0], 'front': [1, -1, 1], 'up': [0, 0, 1]} 
        ]
        imgs = []
        for v in views:
            ctr.set_lookat(v['lookat']); ctr.set_front(v['front']); ctr.set_up(v['up']); ctr.set_zoom(0.35)
            vis.poll_events(); vis.update_renderer()
            imgs.append((255 * np.asarray(vis.capture_screen_float_buffer(False))).astype(np.uint8))
        full_img = np.vstack((np.hstack((imgs[0], imgs[1])), np.hstack((imgs[2], imgs[3]))))
        full_img = cv2.cvtColor(full_img, cv2.COLOR_RGB2BGR)
        full_img = draw_text_overlay(full_img, text_lines)
        return cv2.cvtColor(full_img, cv2.COLOR_BGR2RGB)
    
    frames = []
    def add_pause(frames, last_frame, seconds): 
        for _ in range(int(seconds * fps)): frames.append(last_frame)

    # 1. Downsampling
    frame = capture_frame("1. Input Data (High Res)"); frames.append(frame); add_pause(frames, frame, pause_time)
    params = ctr.convert_to_pinhole_camera_parameters()
    vis.remove_geometry(src_pcd_high, False); vis.remove_geometry(ref_pcd_high, False)
    vis.add_geometry(src_down, False); vis.add_geometry(ref_down, False)
    ctr.convert_from_pinhole_camera_parameters(params)
    frame = capture_frame(f"1. Downsampling (Voxel: {VOXEL_SIZE}m)"); frames.append(frame); add_pause(frames, frame, pause_time)
    
    # 2. RANSAC
    current_T = np.eye(4); attempts = []
    for i in range(9):
        rnd = np.eye(4); rnd[:3, :3] = src_pcd_high.get_rotation_matrix_from_xyz(np.random.uniform(-1, 1, 3))
        rnd[:3, 3] = np.random.uniform(-0.5, 0.5, 3); attempts.append((rnd, f"2. RANSAC Search (Attempt {i+1}/10)"))
    attempts.append((ransac_final_T, "2. RANSAC Found (Attempt 10: Best Match)"))
    for target_T, label in attempts:
        delta_T = np.dot(target_T, np.linalg.inv(current_T))
        src_down.transform(delta_T); vis.update_geometry(src_down); current_T = target_T
        frame = capture_frame([label, "Typical: 100k+ Iterations"]); frames.append(frame); add_pause(frames, frame, 0.5)
    add_pause(frames, frames[-1], 1.0)

    # 3a. Coarse ICP
    src_down.estimate_normals(); ref_down.estimate_normals()
    for i in range(15):
        reg = o3d.pipelines.registration.registration_icp(src_down, ref_down, VOXEL_SIZE * 0.4, np.eye(4), o3d.pipelines.registration.TransformationEstimationPointToPlane(), o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1))
        src_down.transform(reg.transformation); vis.update_geometry(src_down); current_T = np.dot(reg.transformation, current_T)
        frame = capture_frame(f"3a. Coarse ICP (Low Res) - Iter {i+1}"); frames.append(frame)
    add_pause(frames, frames[-1], 1.0)

    # 3b. Fine ICP
    src_pcd_high.transform(current_T)
    params = ctr.convert_to_pinhole_camera_parameters()
    vis.remove_geometry(src_down, False); vis.remove_geometry(ref_down, False)
    vis.add_geometry(src_pcd_high, False); vis.add_geometry(ref_pcd_high, False)
    ctr.convert_from_pinhole_camera_parameters(params)
    frame = capture_frame("3b. Switch to High Res for Fine Tuning"); frames.append(frame); add_pause(frames, frame, pause_time)
    
    src_pcd_high.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=VOXEL_SIZE*2, max_nn=30))
    ref_pcd_high.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=VOXEL_SIZE*2, max_nn=30))
    threshold_fine = VOXEL_SIZE * 0.4
    for i in range(15):
        reg = o3d.pipelines.registration.registration_icp(src_pcd_high, ref_pcd_high, threshold_fine, np.eye(4), o3d.pipelines.registration.TransformationEstimationPointToPlane(), o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1))
        src_pcd_high.transform(reg.transformation); vis.update_geometry(src_pcd_high); current_T = np.dot(reg.transformation, current_T)
        frame = capture_frame(f"3b. Fine ICP (High Res) - Iter {i+1}"); frames.append(frame)

    # 4. Final Stats
    try:
        T_inv = np.linalg.inv(current_T); rot_err = get_angular_error(T_inv[:3,:3], gt_matrix[:3,:3]); trans_err = get_translation_error(T_inv[:3,3], gt_matrix[:3,3])
    except: rot_err, trans_err = 999.0, 999.0
    mat_str = np.array2string(current_T, precision=3, suppress_small=True, separator=', ')
    stats_text = ["ALIGNMENT COMPLETE", f"Rotation Error: {rot_err:.4f} deg", f"Translation Error: {trans_err:.6f} m", "Final Matrix:"] + mat_str.split('\n')
    final_frame = capture_frame(stats_text)
    for _ in range(int(10.0 * fps)): frames.append(final_frame)

    vis.destroy_window()
    
    if save_gif and frames:
        gif_path = os.path.join(DATA_ROOT, scan_folder, f"presentation_{source_id}.gif")
        imageio.mimsave(gif_path, frames, fps=fps, loop=0)
        print(f"GIF Saved: {gif_path}")
    
    if save_mp4 and frames:
        mp4_path = os.path.join(DATA_ROOT, scan_folder, f"presentation_{source_id}.mp4")
        imageio.mimsave(mp4_path, frames, fps=fps, quality=8, macro_block_size=None)
        print(f"Video Saved: {mp4_path}")
        display(Markdown(f"**Video Saved:** {mp4_path}"))

In [ ]:
# Example: Visualize Overlay
# view_alignment_overlay("Test_1/cam_d435/setup_front", source_id=5)

# Example: Create Animation
# create_presentation_gif("Test_1/cam_d435/setup_front", source_id=5)